# Analisis LOGIKA UI 2026

Notebook ini membaca hasil analisis yang telah dijalankan. Gunakan `python scripts/analyze.py` untuk membangun ulang seluruh tabel, fitur, dan baseline. Definisi data tersedia pada `data/raw/DATA_DICTIONARY.md`. Kredensial tidak diperlukan untuk membaca hasil lokal.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
if not (ROOT / 'artifacts').exists():
    ROOT = ROOT.parent
ART = ROOT / 'artifacts'
summary = json.loads((ART / 'summary.json').read_text())
features = pd.read_parquet(ROOT / 'data/processed/application_features.parquet')
train = features[features['split'].eq('train')].copy()
test = features[features['split'].eq('test')].copy()
display(pd.Series({k: v for k, v in summary.items() if k not in ['benchmarks', 'top_model_gain_features', 'versions']}))

## Struktur dan kualitas

Nilai kosong struktural, misalnya pinjaman ditolak atau utilization tanpa limit, mempunyai arti berbeda dari nol.

In [ ]:
display(pd.read_csv(ART / 'tables/table_inventory.csv'))
display(pd.read_csv(ART / 'tables/history_coverage.csv'))
quality = pd.read_csv(ART / 'tables/column_quality.csv')
display(quality[quality['missing'].gt(0)])
audit = json.loads((ART / 'quality_audit.json').read_text())
display(audit)

## Target, waktu, dan asosiasi

Target 1 berarti mencapai DPD 90 dalam 12 bulan kalender. Asosiasi dan importance tidak membuktikan sebab akibat.

In [ ]:
display(Image(filename=str(ART / 'figures/target_and_time.png')))
display(Image(filename=str(ART / 'figures/risk_signals.png')))
risk = pd.read_csv(ART / 'tables/default_by_group.csv')
display(risk[risk['variable'].eq('annual_interest_rate')])
display(pd.read_csv(ART / 'tables/target_correlations.csv').head(15))

## Distribusi train dan test

Perbedaan marginal yang kecil tidak menjamin hubungan fitur-target identik. Label test tidak tersedia.

In [ ]:
display(Image(filename=str(ART / 'figures/train_test_distributions.png')))
display(pd.read_csv(ART / 'tables/train_test_numeric_drift.csv').head(15))
display(pd.read_csv(ART / 'tables/feature_missingness.csv').sort_values('train_missing_pct', ascending=False).head(20))

## Baseline dan kalibrasi

Parameter tetap; holdout stratified dan temporal. Metrik resmi adalah Kolmogorov-Smirnov (KS), sesuai Evaluation yang diberikan pengguna. Semua angka tabel ini adalah hasil lokal, bukan skor leaderboard. Hasil optimasi terbaru ada pada `artifacts/optimization/`.

In [ ]:
benchmark_path = ART / 'tables/model_benchmarks.csv'
if benchmark_path.exists() and summary['benchmarks']:
    display(pd.read_csv(benchmark_path))
    display(Image(filename=str(ART / 'figures/model_roc.png')))
    display(Image(filename=str(ART / 'figures/calibration.png')))
    display(pd.read_csv(ART / 'tables/feature_importance_gain.csv').head(20))
cv_path = ART / 'optimization/cross_validation_summary.json'
if cv_path.exists():
    cv = json.loads(cv_path.read_text())
    display(Markdown('### Optimasi KS terbaru — validasi silang 3 fold'))
    display(pd.DataFrame({'Baseline': cv['baseline'], 'Baru': cv['selected']}).T[['ks', 'mean_fold_ks', 'roc_auc', 'average_precision']])
    display(cv['selected']['recipe'])

## Pemeriksaan yang dapat dijalankan ulang

Assertion ini memeriksa integritas hasil, batas waktu, serta risiko salah urutan submission.

In [ ]:
assert len(train) == 70000 and len(test) == 30000
assert features['application_id'].is_unique and features['customer_id'].is_unique
assert set(train['customer_id']).isdisjoint(test['customer_id'])
assert train['target'].isin([0, 1]).all() and test['target'].isna().all()
assert audit['submission']['same_id_set_as_test']
for checks in audit.values():
    for name, value in checks.items():
        if name.endswith('_at_or_after_cutoff') or name == 'incomplete_month_at_cutoff':
            assert value == 0, (name, value)
print('Integritas dasar dan batas waktu: OK')

## Template penyusunan submission

Isi `predictions` dengan probabilitas model yang Anda latih. Join dengan validasi one-to-one; kode template tidak mengirim submission.

In [ ]:
def align_submission(predictions):
    sample = pd.read_csv(ROOT / 'data/raw/sample_submission.csv')
    assert predictions['application_id'].is_unique
    assert set(predictions['application_id']) == set(test['application_id'])
    result = sample[['application_id']].merge(
        predictions[['application_id', 'default_probability']],
        on='application_id', how='left', validate='one_to_one', sort=False
    )
    assert result['application_id'].equals(sample['application_id'])
    assert result['default_probability'].notna().all()
    assert result['default_probability'].between(0, 1).all()
    return result

## Laporan lengkap

Buka `artifacts/laporan_analisis.html` untuk laporan dengan seluruh grafik; CSV lengkap tersedia di `artifacts/tables/`.

Reproduksi dari terminal: `python scripts/analyze.py --reuse-features`. Untuk membangun agregasi ulang, hilangkan `--reuse-features`.